# RF Vpi Sweep — Scanning Fabry-Perot Cavity

This notebook runs a frequency / power sweep of an RF source while
recording Fabry-Perot cavity transmission traces on an oscilloscope.
The analysis extracts the modulation index $\beta$ from sideband / carrier
area ratios and fits $V_\pi$ at each RF frequency.

## How to use

1. **Install** both `hardwarelib` and `cavityscope` (editable installs).
2. **Edit the instrument section** below to match your lab setup (COM ports, VISA addresses, etc.).
3. **Adjust the sweep config** if needed (frequencies, powers, windows, …).
4. **Run all cells.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cavityscope.core import SweepConfig
from cavityscope.sweep import run_sweep

## 1. Instrument initialisation

Import **concrete** drivers from `hardwarelib` and configure them for your bench.
The sweep logic only sees the generic protocol interfaces, so you can swap
drivers freely.

In [ ]:
from hardwarelib.oscilloscopes.rigol import RigolDHO4000
from hardwarelib.signal_generators.windfreak import WindfreakSynthHD

SCOPE_RESOURCE  = "TCPIP0::192.168.1.50::INSTR"
SCOPE_CHANNEL   = 1
SCOPE_TIMEOUT   = 15_000        # ms

RF_COM_PORT     = "COM12"
RF_CHANNEL      = 0
RF_TIMEOUT      = 0.25          # s

scope = RigolDHO4000(SCOPE_RESOURCE, timeout_ms=SCOPE_TIMEOUT)
rf    = WindfreakSynthHD(RF_COM_PORT, channel=RF_CHANNEL, timeout_s=RF_TIMEOUT)

## 2. Sweep configuration

Adjust any parameter by passing it to `SweepConfig`.  
The defaults match the original script.

In [ ]:
cfg = SweepConfig(
    rf_frequencies_hz = np.arange(0.4e9, 5.1e9, 0.5e9).tolist(),
    rf_powers_dbm     = np.arange(-20.0, 10.1, 2.0).tolist(),

    cavity_fsr_hz         = 1.5e9,
    carrier_window_hz     = 120e6,
    sideband_window_hz    = 80e6,
    sideband_mode         = "both",

    compute_vpi           = True,
    net_power_offset_db   = 0.0,
    assumed_load_ohm      = 50.0,

    output_dir            = "vpi_sweep_output",
    save_trace_plots      = True,
    save_reference_plots  = True,
    save_raw_traces_csv   = False,
)

## 3. Run the sweep

In [ ]:
try:
    scope.open()
    rf.open()

    data = run_sweep(
        scope=scope,
        rf_source=rf,
        cfg=cfg,
        scope_channel=SCOPE_CHANNEL,
    )
finally:
    try:
        rf.set_output(False)
    except Exception:
        pass
    rf.close()
    scope.close()

## 4. Inspect results

In [ ]:
df_results = data["results"]
df_fits    = data["fits"]

print(f"{len(df_results)} measurement points, {len(df_fits)} frequency fits")
df_fits

In [ ]:
if not df_fits.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(
        df_fits["rf_frequency_hz"] / 1e9,
        df_fits["fit_vpi_v"],
        "o-",
    )
    ax.set(xlabel="RF frequency (GHz)", ylabel="$V_\pi$ (V)", title="$V_\pi$ vs frequency")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()